<a href="https://colab.research.google.com/github/tracymuhoozi/machineLearningAssignment/blob/tracy-model-training/notebooks/02_model_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import userdata
token = userdata.get('GH_TOKEN')

!git clone https://{token}@github.com/tracymuhoozi/machineLearningAssignment.git
%cd machineLearningAssignment

!git config user.name "tracymuhoozi"
!git config user.email "tracymuhoozi5@gmail.com"

!git log --oneline
!ls

Cloning into 'machineLearningAssignment'...
remote: Enumerating objects: 36, done.
remote: Counting objects: 100% (36/36), done.
remote: Compressing objects: 100% (26/26), done.
remote: Total 36 (delta 5), reused 35 (delta 4), pack-reused 0 (from 0)
Receiving objects: 100% (36/36), 3.16 MiB | 11.60 MiB/s, done.
Resolving deltas: 100% (5/5), done.
/content/machineLearningAssignment
d4a06d7 (HEAD -> alouzious-work, origin/alouzious-work, origin/HEAD) Removed team in README
d41a050 Add chart images and project README
79819f0 Add EDA notebook with saved outputs
7651834 Add project structure and assignment work
data  images  notebooks  README.md  reports  requirements.txt  src


In [2]:
%cd /content/machineLearningAssignment
!git checkout -b tracy-model-training

/content/machineLearningAssignment
Switched to a new branch 'tracy-model-training'


In [3]:
!ls data notebooks src reports
!cat requirements.txt

data:
diabetes_prediction_dataset.csv
diabetes_prediction_dataset.csv:Zone.Identifier

notebooks:
eda.ipynb

reports:
report.md

src:
main.py
asttokens==3.0.2
comm==0.2.3
contourpy==1.4.0
cycler==0.12.1
debugpy==1.8.22
executing==2.2.1
fonttools==4.66.0
ipykernel==7.3.0
ipython==9.17.1
ipython_pygments_lexers==1.1.1
jedi==0.20.0
jupyter_client==8.10.0
jupyter_core==5.9.1
kiwisolver==1.5.1
matplotlib==3.11.2
matplotlib-inline==0.2.2
nest-asyncio2==1.7.3
numpy==2.5.3
packaging==26.3
pandas==3.0.6
parso==0.8.7
pexpect==4.9.0
pillow==12.3.0
platformdirs==4.12.1
prompt_toolkit==3.0.53
psutil==7.2.2
ptyprocess==0.7.0
pure_eval==0.2.4
Pygments==2.21.0
pyparsing==3.3.3
python-dateutil==2.9.0.post0
pyzmq==27.2.0
scipy==1.18.1
seaborn==0.13.2
six==1.17.0
stack-data==0.6.3
tornado==6.5.10
traitlets==5.16.1
typing_extensions==4.16.0
wcwidth==0.9.1


In [4]:
%cd /content/machineLearningAssignment
!git checkout -b tracy-model-training
!git rm "data/diabetes_prediction_dataset.csv:Zone.Identifier"
!head -40 src/main.py

/content/machineLearningAssignment
fatal: a branch named 'tracy-model-training' already exists
rm 'data/diabetes_prediction_dataset.csv:Zone.Identifier'


In [5]:
%cd /content/machineLearningAssignment
!git checkout tracy-model-training
!git branch
!git status

/content/machineLearningAssignment
D	data/diabetes_prediction_dataset.csv:Zone.Identifier
Already on 'tracy-model-training'
  alouzious-work
* tracy-model-training
On branch tracy-model-training
Changes to be committed:
  (use "git restore --staged <file>..." to unstage)
	deleted:    data/diabetes_prediction_dataset.csv:Zone.Identifier



In [6]:
!git commit -m "Remove Zone.Identifier file from data folder"

[tracy-model-training 1b3f334] Remove Zone.Identifier file from data folder
 1 file changed, 3 deletions(-)
 delete mode 100644 data/diabetes_prediction_dataset.csv:Zone.Identifier


In [7]:
import pandas as pd, numpy as np, joblib, os
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier

df = pd.read_csv('data/diabetes_prediction_dataset.csv')
print(df.shape, "duplicates:", df.duplicated().sum())
print(df['diabetes'].value_counts(normalize=True))
df = df.drop_duplicates()
print("after dedupe:", df.shape)

X, y = df.drop(columns='diabetes'), df['diabetes']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42)

pre = ColumnTransformer([
    ('num', StandardScaler(), ['age','bmi','HbA1c_level','blood_glucose_level']),
    ('cat', OneHotEncoder(handle_unknown='ignore'), ['gender','smoking_history']),
    ('bin', 'passthrough', ['hypertension','heart_disease']),
])

(100000, 9) duplicates: 3854
diabetes
0    0.915
1    0.085
Name: proportion, dtype: float64
after dedupe: (96146, 9)


In [8]:
models = {
    'Baseline (majority)': DummyClassifier(strategy='most_frequent'),
    'Logistic Regression': LogisticRegression(max_iter=1000, class_weight='balanced'),
    'Decision Tree': DecisionTreeClassifier(max_depth=6, class_weight='balanced', random_state=42),
    'KNN': KNeighborsClassifier(n_neighbors=15),
    'Random Forest': RandomForestClassifier(n_estimators=200, class_weight='balanced', n_jobs=-1, random_state=42),
}
cv = StratifiedKFold(5, shuffle=True, random_state=42)
scoring = ['accuracy','precision','recall','f1','roc_auc']

rows = []
for name, m in models.items():
    pipe = Pipeline([('pre', pre), ('model', m)])
    r = cross_validate(pipe, X_train, y_train, cv=cv, scoring=scoring, n_jobs=-1)
    rows.append({'model': name, **{s: f"{r['test_'+s].mean():.3f} ± {r['test_'+s].std():.3f}" for s in scoring}})
results = pd.DataFrame(rows)
results

,model,accuracy,precision,recall,f1,roc_auc
0,Baseline (majority),0.912 ± 0.000,0.000 ± 0.000,0.000 ± 0.000,0.000 ± 0.000,0.500 ± 0.000
1,Logistic Regression,0.887 ± 0.002,0.432 ± 0.003,0.883 ± 0.010,0.580 ± 0.002,0.962 ± 0.002
2,Decision Tree,0.847 ± 0.010,0.361 ± 0.016,0.941 ± 0.011,0.522 ± 0.015,0.968 ± 0.001
3,KNN,0.960 ± 0.001,0.956 ± 0.007,0.570 ± 0.013,0.714 ± 0.009,0.944 ± 0.003
4,Random Forest,0.968 ± 0.001,0.937 ± 0.008,0.686 ± 0.007,0.792 ± 0.006,0.965 ± 0.001


In [9]:
!git add reports
!git commit -m "Add data split, leakage-safe pipeline, and 5-model cross-validation results"

On branch tracy-model-training
nothing to commit, working tree clean


In [10]:
os.makedirs('reports', exist_ok=True)
results.to_csv('reports/model_comparison.csv', index=False)
!git add reports
!git commit -m "Add 5-model cross-validation comparison table"

[tracy-model-training 6799d45] Add 5-model cross-validation comparison table
 1 file changed, 6 insertions(+)
 create mode 100644 reports/model_comparison.csv


In [11]:
def tune(model, grid):
    pipe = Pipeline([('pre', pre), ('model', model)])
    gs = GridSearchCV(pipe, grid, cv=cv, scoring='f1', n_jobs=-1)
    gs.fit(X_train, y_train)
    print(gs.best_params_, "CV F1:", round(gs.best_score_, 3))
    return gs

gs_rf = tune(RandomForestClassifier(class_weight='balanced', random_state=42, n_jobs=-1),
             {'model__n_estimators': [100, 200], 'model__max_depth': [8, 12, None]})

gs_knn = tune(KNeighborsClassifier(),
              {'model__n_neighbors': [5, 15, 25], 'model__weights': ['uniform', 'distance']})

{'model__max_depth': None, 'model__n_estimators': 200} CV F1: 0.792
{'model__n_neighbors': 5, 'model__weights': 'uniform'} CV F1: 0.721


In [12]:
!git add .
!git commit -m "Tune Random Forest and KNN with GridSearchCV"

On branch tracy-model-training
nothing to commit, working tree clean


In [13]:
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

# 1. Save the tuning results so they can be committed and used in the report
tuning = pd.DataFrame([
    {'model': 'Random Forest', 'best_params': str(gs_rf.best_params_), 'cv_f1': round(gs_rf.best_score_, 3)},
    {'model': 'KNN', 'best_params': str(gs_knn.best_params_), 'cv_f1': round(gs_knn.best_score_, 3)},
])
tuning.to_csv('reports/tuning_results.csv', index=False)

# 2. Final one-time test-set check for both tuned models
for name, gs in [('Random Forest', gs_rf), ('KNN', gs_knn)]:
    pred = gs.best_estimator_.predict(X_test)
    proba = gs.best_estimator_.predict_proba(X_test)[:, 1]
    print(f"=== {name} ===")
    print(classification_report(y_test, pred, digits=3))
    print("ROC-AUC:", round(roc_auc_score(y_test, proba), 3))
    print(confusion_matrix(y_test, pred), "\n")

# 3. Save the best pipeline and demo loading it back (Step 10)
best = gs_rf.best_estimator_
os.makedirs('models', exist_ok=True)
joblib.dump(best, 'models/best_pipeline.joblib')

loaded = joblib.load('models/best_pipeline.joblib')
sample = X_test.iloc[[0]]
print(sample)
print("prediction:", loaded.predict(sample), "probability:", loaded.predict_proba(sample))

=== Random Forest ===
              precision    recall  f1-score   support

           0      0.971     0.996     0.983     17534
           1      0.946     0.691     0.799      1696

    accuracy                          0.969     19230
   macro avg      0.958     0.844     0.891     19230
weighted avg      0.969     0.969     0.967     19230

ROC-AUC: 0.962
[[17467    67]
 [  524  1172]] 

=== KNN ===
              precision    recall  f1-score   support

           0      0.964     0.993     0.978     17534
           1      0.891     0.620     0.731      1696

    accuracy                          0.960     19230
   macro avg      0.928     0.806     0.855     19230
weighted avg      0.958     0.960     0.956     19230

ROC-AUC: 0.9
[[17406   128]
 [  645  1051]] 

       gender   age  hypertension  heart_disease smoking_history    bmi  \
82004  Female  51.0             0              0         current  36.77   

       HbA1c_level  blood_glucose_level  
82004          6.6       

In [14]:
!pip freeze | grep -iE "^(scikit-learn|joblib)==" >> requirements.txt
!git add requirements.txt reports models
!git commit -m "Add tuning results, save best Random Forest pipeline, add sklearn/joblib to requirements"
!git push -u origin tracy-model-training

[tracy-model-training 17d1b68] Add tuning results, save best Random Forest pipeline, add sklearn/joblib to requirements
 3 files changed, 5 insertions(+)
 create mode 100644 models/best_pipeline.joblib
 create mode 100644 reports/tuning_results.csv
Enumerating objects: 18, done.
Counting objects: 100% (18/18), done.
Delta compression using up to 2 threads
Compressing objects: 100% (13/13), done.
Writing objects: 100% (14/14), 24.94 MiB | 4.26 MiB/s, done.
Total 14 (delta 6), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (6/6), completed with 3 local objects.
remote: error: Trace: cfa261741ab2a83d62ff8258fbec41560ceb1decd066acaff8a86147b8adc899
remote: error: See https://gh.io/lfs for more information.
remote: error: File models/best_pipeline.joblib is 124.87 MB; this exceeds GitHub's file size limit of 100.00 MB
remote: error: GH001: Large files detected. You may want to try Git Large File Storage - https://git-lfs.github.com.
To https://github.com/tracymuhoozi/machi

In [15]:
# 1. Undo the local commit
!git reset --soft HEAD~1
!git restore --staged models/best_pipeline.joblib

# 2. Re-save with compression
joblib.dump(best, 'models/best_pipeline.joblib', compress=3)
!ls -lh models

total 27M
-rw-r--r-- 1 root root 27M Sep 29 08:11 best_pipeline.joblib


In [16]:
loaded = joblib.load('models/best_pipeline.joblib')
print(loaded.predict(X_test.iloc[[0]]))   # confirm it still works

!git add models reports requirements.txt
!git commit -m "Add tuning results, save compressed Random Forest pipeline, add sklearn/joblib to requirements"
!git push -u origin tracy-model-training

[0]
[tracy-model-training 94ba202] Add tuning results, save compressed Random Forest pipeline, add sklearn/joblib to requirements
 3 files changed, 5 insertions(+)
 create mode 100644 models/best_pipeline.joblib
 create mode 100644 reports/tuning_results.csv
Enumerating objects: 18, done.
Counting objects: 100% (18/18), done.
Delta compression using up to 2 threads
Compressing objects: 100% (13/13), done.
Writing objects: 100% (14/14), 26.41 MiB | 11.91 MiB/s, done.
Total 14 (delta 6), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (6/6), completed with 3 local objects.
remote: 
remote: Create a pull request for 'tracy-model-training' on GitHub by visiting:
remote:      https://github.com/tracymuhoozi/machineLearningAssignment/pull/new/tracy-model-training
remote: 
To https://github.com/tracymuhoozi/machineLearningAssignment.git
 * [new branch]      tracy-model-training -> tracy-model-training
branch 'tracy-model-training' set up to track 'origin/tracy-model-training'

In [17]:
from sklearn.metrics import recall_score, precision_score, f1_score

best = joblib.load('models/best_pipeline.joblib')
proba = best.predict_proba(X_test)[:, 1]
pred = (proba >= 0.5).astype(int)

ev = X_test.copy()
ev['y_true'], ev['y_pred'], ev['proba'] = y_test.values, pred, proba
ev['age_band'] = pd.cut(ev['age'], [0, 30, 45, 60, 75, 120],
                        labels=['<30', '30-44', '45-59', '60-74', '75+'])

def group_report(col):
    rows = []
    for g, d in ev.groupby(col, observed=True):
        rows.append({col: g, 'n': len(d), 'diabetic_n': int(d.y_true.sum()),
                     'recall': round(recall_score(d.y_true, d.y_pred, zero_division=0), 3),
                     'precision': round(precision_score(d.y_true, d.y_pred, zero_division=0), 3),
                     'f1': round(f1_score(d.y_true, d.y_pred, zero_division=0), 3),
                     'missed': int(((d.y_true == 1) & (d.y_pred == 0)).sum())})
    return pd.DataFrame(rows)

subgroups = {c: group_report(c) for c in ['age_band', 'gender', 'smoking_history', 'hypertension', 'heart_disease']}
for c, t in subgroups.items():
    print(t, "\n")

# What do the missed cases look like versus the caught ones?
fn = ev[(ev.y_true == 1) & (ev.y_pred == 0)]
tp = ev[(ev.y_true == 1) & (ev.y_pred == 1)]
print(pd.DataFrame({'missed (FN)': fn[['age','bmi','HbA1c_level','blood_glucose_level']].mean(),
                    'caught (TP)': tp[['age','bmi','HbA1c_level','blood_glucose_level']].mean()}).round(2))

# Threshold trade-off
thr = []
for t in [0.5, 0.4, 0.3, 0.25, 0.2]:
    p = (proba >= t).astype(int)
    thr.append({'threshold': t, 'recall': round(recall_score(y_test, p), 3),
                'precision': round(precision_score(y_test, p), 3),
                'f1': round(f1_score(y_test, p), 3),
                'false_neg': int(((y_test == 1) & (p == 0)).sum()),
                'false_pos': int(((y_test == 0) & (p == 1)).sum())})
threshold_table = pd.DataFrame(thr)
print(threshold_table)

os.makedirs('reports', exist_ok=True)
pd.concat(subgroups.values(), keys=subgroups.keys()).to_csv('reports/subgroup_analysis.csv')
threshold_table.to_csv('reports/threshold_analysis.csv', index=False)

  age_band     n  diabetic_n  recall  precision     f1  missed
0      <30  6493          72   0.653      0.979  0.783      25
1    30-44  3815         180   0.644      0.975  0.776      64
2    45-59  4318         520   0.683      0.949  0.794     165
3    60-74  3089         618   0.725      0.926  0.813     170
4      75+  1515         306   0.683      0.933  0.789      97 

   gender      n  diabetic_n  recall  precision     f1  missed
0  Female  11344         929   0.695      0.943  0.800     283
1    Male   7882         767   0.690      0.938  0.795     238
2   Other      4           0   0.000      0.000  0.000       0 

  smoking_history     n  diabetic_n  recall  precision     f1  missed
0         No Info  6580         294   0.707      0.929  0.803      86
1         current  1848         213   0.718      0.968  0.825      60
2            ever   831         102   0.539      0.982  0.696      47
3          former  1823         309   0.676      0.917  0.778     100
4           neve

In [18]:
!git add reports
!git commit -m "Add error, subgroup and threshold analysis for best model"
!git push

[tracy-model-training 4c63dbe] Add error, subgroup and threshold analysis for best model
 2 files changed, 25 insertions(+)
 create mode 100644 reports/subgroup_analysis.csv
 create mode 100644 reports/threshold_analysis.csv
Enumerating objects: 7, done.
Counting objects: 100% (7/7), done.
Delta compression using up to 2 threads
Compressing objects: 100% (5/5), done.
Writing objects: 100% (5/5), 1.09 KiB | 186.00 KiB/s, done.
Total 5 (delta 1), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (1/1), completed with 1 local object.
To https://github.com/tracymuhoozi/machineLearningAssignment.git
   94ba202..4c63dbe  tracy-model-training -> tracy-model-training
